# PII detection: redact, mask, hash, block

PII middleware changes or blocks selected content at the model boundary. Here all addresses are fictional. It is not a complete data-loss-prevention system: original notebook inputs and other logging layers may still hold the source text.

[PII detection](https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.tools import get_contact
from langchain.agents import create_agent
from langchain.agents.middleware import PIIDetectionError, PIIMiddleware
from model_config import get_model, run_config
import re
from lab.faults import ScriptedModel
from langchain_core.messages import AIMessage


In [ ]:
prompt("pii")

In [ ]:
model = get_model()

In [ ]:
# Execute
strategy = "redact"
agent = create_agent(model=model, middleware=[
    PIIMiddleware("email", strategy=strategy),
    TraceMiddleware(f"06_{strategy}", pretty=True),
])
redacted_result = agent.invoke({"messages": [("user", prompt("pii"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(redacted_result["messages"], pretty=True)

In [ ]:
# Execute
strategy = "mask"
agent = create_agent(model=model, middleware=[
    PIIMiddleware("email", strategy=strategy),
    TraceMiddleware(f"06_{strategy}", pretty=True),
])
masked_result = agent.invoke({"messages": [("user", prompt("pii"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(masked_result["messages"], pretty=True)

In [ ]:
# Execute
strategy = "hash"
agent = create_agent(model=model, middleware=[
    PIIMiddleware("email", strategy=strategy),
    TraceMiddleware(f"06_{strategy}", pretty=True),
])
hashed_result = agent.invoke({"messages": [("user", prompt("pii"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(hashed_result["messages"], pretty=True)

### Block before the model request

In [ ]:
trace = TraceMiddleware("06_block", pretty=True)

agent = create_agent(model=model, middleware=[PIIMiddleware("email", strategy="block"), trace])

with expect_error(PIIDetectionError):
    agent.invoke({"messages": [("user", prompt("pii"))]}, config=run_config())
assert not any(event["kind"] == "MODEL REQUEST" for event in trace.events)

### Apply the policy to tool results as well
Inspect the next model request to see the transformed contact value. A trace closer to the tool may still record the original fictional address.

In [ ]:
# Execute
agent = create_agent(model=model, tools=[get_contact], middleware=[
    PIIMiddleware("email", strategy="redact", apply_to_input=True, apply_to_output=True, apply_to_tool_results=True),
    TraceMiddleware("06_tool_pii", pretty=True),
])
tool_pii_result = agent.invoke({"messages": [("user", prompt("contact"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(tool_pii_result["messages"], pretty=True)

### Custom regex and callable detectors

Built-in detectors cover email, credit cards, IP addresses, MAC addresses and URLs. A custom type accepts a regex or a callable returning matches with `text`, `start` and `end`. Input, assistant output and tool results can be filtered independently. With output filtering enabled, streaming output is filtered as well.

In [ ]:
# Execute
def employee_ids(text):
    return [{"text": match.group(), "start": match.start(), "end": match.end()} for match in re.finditer(r"EMP-\d{4}", text)]

custom_pii_agent = create_agent(model=ScriptedModel(responses=[AIMessage(content="Employee EMP-2048 has API key sk-demo-1234.")]), middleware=[
    PIIMiddleware("employee_id", detector=employee_ids, strategy="redact", apply_to_input=True, apply_to_output=True),
    PIIMiddleware("api_key", detector=r"sk-demo-\d+", strategy="redact", apply_to_input=True, apply_to_output=True),
])
custom_pii_result = custom_pii_agent.invoke({"messages": [("user", "Check EMP-2048 and sk-demo-1234.")]}, config=run_config())

In [ ]:
# Inspect output
show_messages(custom_pii_result["messages"], pretty=True)
assert "EMP-2048" not in custom_pii_result["messages"][-1].content
assert "sk-demo-1234" not in custom_pii_result["messages"][-1].content